## Load site column information

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

SITES_DIR = Path("../src/data/sites")

records = []
for parquet_path in sorted(SITES_DIR.glob("*.parquet")):
    schema = pq.read_schema(parquet_path)
    parquet_columns = list(schema.names)
    columns_to_read = [c for c in ["ee_site_id", "Circuit Label", "End Use"] if c in parquet_columns]

    table = pq.read_table(parquet_path, columns=columns_to_read)
    df = table.to_pandas()

    ee_site_id = (
        str(df["ee_site_id"].dropna().iloc[0])
        if "ee_site_id" in df.columns and not df["ee_site_id"].dropna().empty
        else parquet_path.stem
    )
    circuit_labels = (
        sorted(df["Circuit Label"].dropna().astype(str).unique().tolist())
        if "Circuit Label" in df.columns
        else []
    )
    end_uses = (
        sorted(df["End Use"].dropna().astype(str).unique().tolist())
        if "End Use" in df.columns
        else []
    )
    label_end_use_pairs = (
        df[["Circuit Label", "End Use"]]
        .dropna()
        .drop_duplicates()
        .astype(str)
        .sort_values(["Circuit Label", "End Use"])
        .to_dict("records")
        if {"Circuit Label", "End Use"}.issubset(df.columns)
        else []
    )

    records.append(
        {
            "ee_site_id": ee_site_id,
            "file": parquet_path.name,
            "parquet_columns": parquet_columns,
            "circuit_labels": circuit_labels,
            "end_uses": end_uses,
            "label_end_use_pairs": label_end_use_pairs,
            "n_circuit_labels": len(circuit_labels),
            "n_end_uses": len(end_uses),
        }
    )

site_column_info = pd.DataFrame(records)
all_circuit_labels = sorted(
    {label for labels in site_column_info["circuit_labels"] for label in labels}
)
all_end_uses = sorted({end_use for end_uses in site_column_info["end_uses"] for end_use in end_uses})
all_label_end_use_pairs = sorted(
    {
        (pair["Circuit Label"], pair["End Use"])
        for pairs in site_column_info["label_end_use_pairs"]
        for pair in pairs
    }
)

print(f"Loaded column information for {len(site_column_info)} site files")
print(f"Unique circuit labels: {len(all_circuit_labels)}")
print(f"Unique end uses: {len(all_end_uses)}")
print(f"Unique circuit label/end-use pairs: {len(all_label_end_use_pairs)}")
site_column_info.head()

In [ ]:
all_circuit_labels

In [ ]:
all_label_end_use_pairs

In [ ]:
pair_counts = (
    site_column_info[["ee_site_id", "label_end_use_pairs"]]
    .explode("label_end_use_pairs")
    .dropna(subset=["label_end_use_pairs"])
)
pair_counts[["Circuit Label", "End Use"]] = pd.DataFrame(
    pair_counts["label_end_use_pairs"].tolist(),
    index=pair_counts.index,
)

pair_counts = (
    pair_counts
    .groupby(["Circuit Label", "End Use"])
    .agg(n_sites=("ee_site_id", "nunique"))
    .reset_index()
    .sort_values(["n_sites", "Circuit Label", "End Use"], ascending=[False, True, True])
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(pair_counts)

print(f"Unique circuit-label/end-use pairs: {len(pair_counts)}")


,Circuit Label,End Use,n_sites
56,Other_1,Other,251
67,Other_2,Other,249
6,Clothes Dryer_1,Clothes Dryer,246
69,Other_3,Other,241
85,Stove/Oven/Range_1,Stove/Oven/Range,239
70,Other_4,Other,233
44,Mains_1,Mains,232
71,Other_5,Other,220
72,Other_6,Other,209
8,Clothes Washer_1,Clothes Washer,201


Unique circuit-label/end-use pairs: 89


## Build calculated overall time series

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

SITES_DIR = Path("../src/data/sites")
CATEGORY_MAP_PATH = SITES_DIR / "end_use_category_map.csv"
OUT_PATH = SITES_DIR / "calculated_overall_timeseries.parquet"

category_map = pd.read_csv(CATEGORY_MAP_PATH)
category_by_end_use = category_map.set_index("end_use")["category"]
excluded_categories = {"Renewable", "Overall"}
required_columns = ["ee_site_id", "min_t", "max_t", "power", "End Use", "state", "hz", "cz"]

site_timeseries = []
missing_end_uses = set()

site_files = sorted(
    p
    for p in SITES_DIR.glob("*.parquet")
    if p.name != OUT_PATH.name and "End Use" in pq.read_schema(p).names
)

for i, parquet_path in enumerate(site_files, start=1):
    schema_columns = pq.read_schema(parquet_path).names
    columns_to_read = [c for c in required_columns if c in schema_columns]
    df = pq.read_table(parquet_path, columns=columns_to_read).to_pandas()

    df["category"] = df["End Use"].map(category_by_end_use)
    missing_end_uses.update(df.loc[df["category"].isna(), "End Use"].dropna().astype(str).unique())

    included = df[~df["category"].isin(excluded_categories)].copy()
    group_columns = [c for c in ["ee_site_id", "min_t", "max_t", "state", "hz", "cz"] if c in included.columns]

    calculated = (
        included
        .groupby(group_columns, as_index=False, dropna=False)
        .agg(calculated_overall_power=("power", "sum"))
    )
    calculated["source_site_file"] = parquet_path.name
    site_timeseries.append(calculated)

    if i % 25 == 0 or i == len(site_files):
        print(f"Processed {i}/{len(site_files)} site files")

if missing_end_uses:
    raise ValueError(f"Missing end-use categories: {sorted(missing_end_uses)}")

calculated_overall_timeseries = (
    pd.concat(site_timeseries, ignore_index=True)
    .sort_values(["ee_site_id", "min_t"])
    .reset_index(drop=True)
)

calculated_overall_timeseries.to_parquet(OUT_PATH, index=False)
print(f"Wrote {len(calculated_overall_timeseries):,} rows for {calculated_overall_timeseries['ee_site_id'].nunique():,} sites -> {OUT_PATH}")
calculated_overall_timeseries.head()


In [ ]:
negative_overall = calculated_overall_timeseries[
    calculated_overall_timeseries["calculated_overall_power"] < 0
]

negative_site_summary = (
    negative_overall
    .groupby("ee_site_id")["calculated_overall_power"]
    .agg(negative_rows="count", min_power="min", mean_negative_power="mean")
    .sort_values("min_power")
)

print(
    f"Negative calculated overall rows: {len(negative_overall):,} "
    f"across {negative_overall['ee_site_id'].nunique():,} sites"
)
display(negative_site_summary)

negative_source_rows = []
for site_id in negative_site_summary.index:
    site_path = SITES_DIR / f"{site_id}.parquet"
    raw_site = pd.read_parquet(site_path)
    raw_site["category"] = raw_site["End Use"].map(category_by_end_use)
    included_site = raw_site[~raw_site["category"].isin(excluded_categories)].copy()
    negative_source_rows.append(
        included_site[included_site["power"] < 0]
        .groupby(["End Use", "category"], dropna=False)["power"]
        .agg(negative_rows="count", min_power="min", mean_power="mean", total_power="sum")
        .assign(ee_site_id=site_id)
        .reset_index()
    )

negative_source_summary = (
    pd.concat(negative_source_rows, ignore_index=True)
    .set_index(["ee_site_id", "End Use", "category"])
    .sort_values("total_power")
)
negative_source_summary


## Plot sampled calculated overall time series

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

COMBINED_PATH = Path("../src/data/sites/calculated_overall_timeseries.parquet")

calculated_overall_timeseries = pd.read_parquet(COMBINED_PATH)
calculated_overall_timeseries["min_t"] = pd.to_datetime(calculated_overall_timeseries["min_t"])

sample_site_ids = (
    calculated_overall_timeseries["ee_site_id"]
    .drop_duplicates()
    .sample(n=min(5, calculated_overall_timeseries["ee_site_id"].nunique()), random_state=42)
    .tolist()
)

fig, ax = plt.subplots(figsize=(14, 6))
for site_id in sample_site_ids:
    site_ts = (
        calculated_overall_timeseries[calculated_overall_timeseries["ee_site_id"] == site_id]
        .sort_values("min_t")
    )
    ax.plot(
        site_ts["min_t"],
        site_ts["calculated_overall_power"],
        linewidth=0.8,
        alpha=0.85,
        label=f"Site {site_id}",
    )

ax.set_title("Calculated overall power for 5 sampled sites")
ax.set_xlabel("Time")
ax.set_ylabel("kW")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

sample_site_ids
